In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import re

In [ ]:
df = pd.read_csv('./Data/standardized_data.csv')
df

In [ ]:
df.groupby('spot').count().sort_values('year', ascending=False).head(30)

In [ ]:
startdate = pd.to_datetime(df['startdate'])
months = pd.period_range(
    startdate.min().to_period('M'), startdate.max().to_period('M'), freq='M'
)
monthly_counts = (
    df.assign(month_year=startdate.dt.to_period('M'))
    .groupby(['spot', 'month_year'])
    .size()
)
top_spots = df['spot'].value_counts().head(20).index
positions = np.arange(len(months))

for spot in top_spots:
    counts = monthly_counts.reindex(
        pd.MultiIndex.from_product([[spot], months]), fill_value=0
    ).to_numpy()

    fig, ax = plt.subplots(figsize=(18, 5))
    ax.bar(positions, counts, edgecolor='black')
    ax.set_xticks(positions)
    ax.set_xticklabels(
        [month.strftime('%m-%y') for month in months], rotation=45
    )
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)

    for i, month in enumerate(months):
        if i > 0 and month.month == 1:
            ax.axvline(i - 0.5, color='black', linestyle='--')

    ax.set_xlabel('month-year')
    ax.set_ylabel('count')
    ax.set_title(spot)
    fig.tight_layout()
    plt.show()

In [ ]:
pivot_pct

In [ ]:
pivot

In [ ]:
pivot_pct

In [ ]:
df

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('./Data/standardized_data.csv')

df = df[df['spot'] != 'The Hatchery']


# 1. Extract month order (Jan–Dec)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
# df['Month'] = pd.Categorical(df['Date'].str[:3], categories=months, ordered=True)

# 2. Keep top 20 most active spots (excluding blank/[Secret spot] if desired)
top_spots = df['spot'].value_counts().head(20).index
pivot = (
    df[df['spot'].isin(top_spots)]
    .pivot_table(index='spot', columns='month', values='id', aggfunc='count', fill_value=0, observed=False)
    .loc[top_spots]  # preserves ranking from most active at top to least at bottom
)
pivot = pivot.reindex(columns=range(1, 13), fill_value=0)
pivot.columns = months

# # 3. Normalize each row to % of that spot's total tracks
row_totals = pivot.sum(axis=1)
pivot_pct = pivot.div(row_totals, axis=0) * 100
pivot_pct.index = [f"{spot} (n={n:,})" for spot, n in row_totals.items()]

# 3. Plot heatmap
plt.figure(figsize=(12, 8))
ax = sns.heatmap(pivot_pct, cmap='YlGnBu', annot=True, linewidths=0.5, cbar_kws={'label': 'Total GPS Tracks'}, vmax=30)
ax.set_xticklabels(months, rotation=45, ha='right')
plt.title('Wingfoil Activity [% of Total Entries] by Spot & Month (2021–2026)')
plt.xlabel('Month')
plt.ylabel('Spot')
plt.tight_layout()
plt.show()
